# 08 · Studi Kasus Curah Hujan (Data Terbuka) — Bab 9

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 9: prediksi hujan harian titik lokasi/grid (data terbuka: CHIRPS/ERA5-Land) — dua lintasan (regresi mm dan klasifikasi kategori), fitur ERA5-Land + indeks iklim, baseline klimatologi, GRU multivariate, walk-forward, verifikasi CSI/POD/FAR dengan threshold, dan interpretasi permutation importance.

## 1. Setup & Data Contoh

Notebook wajib memakai **data nyata terbuka** (ERA5-Land + CHIRPS + ONI/MEI/RMM, via `scripts/download_*.py`); barat = jakarta, timur = kupang. Data sintetik tidak dipakai; bila file belum tersedia, notebook berhenti dengan instruksi unduh.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

np.random.seed(42)
tf.random.set_seed(42)

def _buscar_raiz():
    """Cari pasta repo buku yang memuat manuscripts/ (portabel Colab+local)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "manuscripts").exists():
            return p
    return Path.cwd()

_BASE = _buscar_raiz() / "manuscripts/ch-09-studi-kasus-curah-hujan-terbuka/data"
_RAW = _BASE / "raw"

def _idx_iklim(idx):
    """Indeks iklim: RMM harian (BoM) & ONI monthly-anom (CPC), ffill ke harian."""
    for req in ("indeks_rmm.csv", "indeks_oni.csv"):
        if not (_RAW / req).exists():
            raise FileNotFoundError(
                f"Indeks iklim {req} belum tersedia. Jalankan:\\n"
                "  python scripts/download_indices.py"
            )
    rmm = pd.read_csv(_RAW / "indeks_rmm.csv", parse_dates=["tanggal"]).set_index("tanggal")
    rmm = rmm[~rmm.index.duplicated(keep="first")]
    rmm1 = rmm["rmm1"].reindex(idx, method="ffill").fillna(0.0).values
    oni = pd.read_csv(_RAW / "indeks_oni.csv", parse_dates=["tanggal"]).set_index("tanggal")
    oni = oni[~oni.index.duplicated(keep="first")]
    nino = oni["anom"].reindex(idx, method="ffill").fillna(0.0).values
    return rmm1, nino

def _era5land(nombre):
    """Concat data harian ERA5-Land nyata (tp_mm/t2m_c) bila tersedia."""
    fs = sorted((_BASE / "era5" / nombre).glob(f"era5land_{nombre}_*_daily.csv"))
    if not fs:
        return None
    era = pd.concat([pd.read_csv(p, parse_dates=["tanggal"]).set_index("tanggal") for p in fs])
    return era[~era.index.duplicated(keep="first")]

# ---- BARAT: jakarta (data nyata terbuka)
era_w = _era5land("jakarta")
if era_w is None:
    raise FileNotFoundError(
        "Data nyata ERA5-Land jakarta belum tersedia. Jalankan:\\n"
        "  python scripts/download_era5.py --stations jakarta --years 2010-2025 --land --process"
    )
t = era_w.index
df = pd.DataFrame({"r_hujan": era_w["tp_mm"], "suhu": era_w["t2m_c"]}, index=t)
df["rmm1"], df["nino34"] = _idx_iklim(t)
ETIQ_W = "barat (jakarta): data nyata ERA5-Land (+ CHIRPS opsional)"

# ---- TIMUR: kupang (data nyata terbuka)
era_e = _era5land("kupang")
if era_e is None:
    raise FileNotFoundError(
        "Data nyata ERA5-Land kupang belum tersedia. Jalankan:\\n"
        "  python scripts/download_era5.py --stations kupang --years 2010-2025 --land --process"
    )
te = era_e.index
df_east = pd.DataFrame({"r_hujan": era_e["tp_mm"], "suhu": era_e["t2m_c"]}, index=te)
df_east["rmm1"], df_east["nino34"] = _idx_iklim(te)
ETIQ_E = "timur (kupang): data nyata ERA5-Land"

print("WEST:", ETIQ_W)
print(df.head())
print("Distribusi kategori lebat (>=50):", int((df["r_hujan"]>=50).sum()), "hari")
print("\nEAST:", ETIQ_E)
print(df_east.head())
print("Distribusi kategori lebat east (>=50):", int((df_east["r_hujan"]>=50).sum()), "hari")

## 2. Feature Engineering (Tabel 9.1)

Lag, musiman sinus, indeks iklim.

In [2]:
for lag in [1,2,3,7]:
    df[f"hujan_t{lag}"] = df["r_hujan"].shift(lag)
df["mus_sin"] = np.sin(2*np.pi*df.index.dayofyear/365.25)
df["mus_cos"] = np.cos(2*np.pi*df.index.dayofyear/365.25)
feat = [c for c in df.columns if c != "r_hujan"]
print("Fitur:", feat)

# target kategori (Tabel 9.2: <20 = 0, 20-<50 = 1, >=50 = 2)
kategori = np.select([df["r_hujan"]<20, df["r_hujan"]<50], [0,1], default=2)
df["kategori"] = kategori.astype(int)

Fitur: ['suhu', 'rmm1', 'nino34', 'hujan_t1', 'hujan_t2', 'hujan_t3', 'hujan_t7', 'mus_sin', 'mus_cos']


## 2b · Fitur ERA5 (Catatan 9.2): lag minimal 1 hari + konversi satuan

ERA5 `total_precipitation` dalam meter (akumulasi/jam) → konversi mm + agregasi harian 07.00-07.00; fitur di-lag minimal 1 hari.

In [3]:
# Fitur ERA5: memakai data nyata harian hasil scripts/download_era5.py --land --process
# -> era5land_<stasiun>_<thn>_daily.csv (kolom tp_mm/t2m_c/u10/v10; agregasi harian
# sudah diterapkan di skrip: tp akumulasi m -> mm, t2m K -> degC).
era0 = pd.read_csv(_BASE / "era5" / "jakarta" / "era5land_jakarta_2018_daily.csv",
                   parse_dates=["tanggal"]).set_index("tanggal")
print("ERA5-Land nyata (jakarta) kolom:", list(era0.columns))
print("tp_mm (mm/hari), lag 1 hari:", era0["tp_mm"].shift(1).dropna().head(3).round(2).values)


## 3. Baseline Klimatologi "Cerdas"

Rata-rata per hari Julian dari data latih, lalu ulangi ke test.

In [4]:
def mae(a,b): return float(np.mean(np.abs(a-b)))

df_ml = df.dropna().copy()
n = len(df_ml)
ntr = int(n*0.7); nva = int(n*0.15)

df_tr = df_ml.iloc[:ntr]
klim = df_tr.groupby(df_tr.index.dayofyear)["r_hujan"].mean()
base_test = klim.reindex(df_ml.index[ntr+nva:].dayofyear).fillna(0).values
y_test = df_ml["r_hujan"].values[ntr+nva:]
print("MAE klimatologi:", round(mae(y_test, base_test), 4))

# Baseline tambahan (tujuan Bab 9): persistence (lag 1 hari) & ARIMA singkat
pers = df_ml["r_hujan"].shift(1)
pers_test = pers.values[ntr + nva:]
print("MAE persistence (lag 1 hari):", round(mae(y_test, pers_test), 4))

try:
    from statsmodels.tsa.arima.model import ARIMA
    arima = ARIMA(df_ml["r_hujan"].iloc[:ntr].values, order=(1, 0, 1)).fit()
    f_ar = np.maximum(arima.forecast(len(y_test)), 0.0)
    print("MAE ARIMA(1,0,1) singkat:", round(mae(y_test, f_ar), 4))
except Exception as e:
    print("[info] ARIMA singkat tidak diexecute:", e)


MAE klimatologi: 9.5351


## 4. Normalisasi (skala latih) + Windowing untuk GRU

In [5]:
from sklearn.preprocessing import StandardScaler

w = 14
sc = StandardScaler().fit(df_ml[feat].iloc[:ntr])
Z = sc.transform(df_ml[feat])
Y = df_ml["r_hujan"].values
K = df_ml["kategori"].values

def buat_window(X, y, w=14):
    Xw, yw = [], []
    for i in range(len(X) - w):
        Xw.append(X[i:i+w]); yw.append(y[i+w])
    return np.array(Xw), np.array(yw)

Xw, Yw = buat_window(Z, Y, w)
_, Kw = buat_window(Z, K, w)
n2 = len(Xw)
ntr2, nva2 = int(n2*0.7), int(n2*0.15)
Xtr, Xva, Xte = Xw[:ntr2], Xw[ntr2:ntr2+nva2], Xw[ntr2+nva2:]
Ytr, Yva, Yte = Yw[:ntr2], Yw[ntr2:ntr2+nva2], Yw[ntr2+nva2:]
Ktr, Kva, Kte = Kw[:ntr2], Kw[ntr2:ntr2+nva2], Kw[ntr2+nva2:]
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

train (1519, 14, 9) val (325, 14, 9) test (326, 14, 9)


## 4b. Walk-Forward 3 Blok (GRU, window bergerak)

Pattern Bab 5 §5.5: window latih yang bergerak (expanding), evaluasi blok oleh blok,
dibandingkan dengan baseline klimatologi (per hari Julian) pada blok test yang sejajar.

In [ ]:
# Walk-forward 3 blok pada data yang sampai sudah windowed (§4)
n3 = len(Xw)
blok3 = n3 // 4
folds_wf, klim_wf = [], []
for b in range(1, 4):
    i_end = b * blok3
    if i_end + blok3 > n3:
        break
    mw = tf.keras.Sequential([
        tf.keras.layers.GRU(16, input_shape=(w, Xtr.shape[2])),
        tf.keras.layers.Dense(1)])
    mw.compile(optimizer="adam", loss="mse", metrics=["mae"])
    mw.fit(Xw[:i_end], np.log1p(Yw[:i_end]), epochs=12, batch_size=32, verbose=0)
    p = np.expm1(mw.predict(Xw[i_end:i_end + blok3], verbose=0).ravel())
    folds_wf.append(mae(Yw[i_end:i_end + blok3], p))
    # klimatologi sejajar: target Yw[i] <-> df_ml iloc i+w
    rows = df_ml.index[i_end + w:i_end + blok3 + w]
    klim_wf.append(mae(Yw[i_end:i_end + blok3],
                       klim.reindex(rows.dayofyear).fillna(0).values))
    print(f"  blok {b}: MAE_GRU={folds_wf[-1]:.4f}  MAE_klima={klim_wf[-1]:.4f}")

if folds_wf:
    print("Walk-forward 3 blok (GRU) rata-rata: %.4f | klimatologi rata-rata: %.4f | "
          "skill = %+.3f" % (np.mean(folds_wf), np.mean(klim_wf),
                             1 - np.mean(folds_wf) / np.mean(klim_wf)))

## 5. Regresi (GRU, transformasi log1p)

In [6]:
bias = 1.0
m = tf.keras.Sequential([
    tf.keras.layers.GRU(16, input_shape=(w, Xtr.shape[2])),
    tf.keras.layers.Dense(1)])
m.compile(optimizer="adam", loss="mse", metrics=["mae"])
m.fit(Xtr, np.log1p(Ytr), validation_data=(Xva, np.log1p(Yva)),
      epochs=30, batch_size=32, verbose=0)
pred_log = m.predict(Xte, verbose=0).ravel()
pred = np.expm1(pred_log)
print("MAE GRU (mm, skala asli):", round(mae(Yte, pred), 4))

# klimatologi untuk test yang sejajar dengan pred (potong w; allineat ke index windowed)
i_te = ntr2 + nva2 + w          # baris df_ml pertama yang menjadi target test
klim_test = klim.reindex(df_ml.index[i_te:i_te+len(Yte)].dayofyear).fillna(0).values
print("MAE klima (sejajar):", round(mae(Yte, klim_test), 4))


C:\Users\Hi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


MAE GRU (mm, skala asli): 7.1885
MAE klima (sejajar): 9.4178


## 6. Klasifikasi Biner (Lebat vs Tidak) + Threshold (Kode 9.1)

In [7]:
leb = (Kte >= 2).astype(int)   # lebat sebagai kelas positif
w_lap = {0: 1.0, 1: 8.0}

mc = tf.keras.Sequential([
    tf.keras.layers.GRU(16, input_shape=(w, Xtr.shape[2])),
    tf.keras.layers.Dense(1, activation="sigmoid")])
mc.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
mc.fit(Xtr, (Ktr>=2).astype(int), validation_data=(Xva, (Kva>=2).astype(int)),
      epochs=30, batch_size=32, class_weight=w_lap, verbose=0)
prob = mc.predict(Xte, verbose=0).ravel()

def verifikasi(y_true, prob, thresholds=[0.2, 0.4, 0.5, 0.6, 0.8]):
    baris = []
    for t in thresholds:
        yp = (prob>=t).astype(int)
        tp=((yp==1)&(y_true==1)).sum(); fp=((yp==1)&(y_true==0)).sum(); fn=((yp==0)&(y_true==1)).sum()
        pod=tp/(tp+fn) if tp+fn else 0; far=fp/(tp+fp) if tp+fp else 1
        csi=tp/(tp+fp+fn) if tp+fp+fn else 0
        baris.append((t, pod, far, csi))
    return pd.DataFrame(baris, columns=["threshold","POD","FAR","CSI"])

verifikasi(leb, prob).round(3)

C:\Users\Hi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


,threshold,POD,FAR,CSI
0,0.2,1.0,0.949,0.051
1,0.4,0.5,0.935,0.061
2,0.5,0.5,0.913,0.080
3,0.6,0.5,0.846,0.133
4,0.8,0.0,1.000,0.000


### Kurva Precision-Recall untuk hujan lebat (Gambar 9.1)

Untuk kelas langka, kurva PR lebih jujur daripada ROC/AUC (Bab 3 §3.9). **Baseline kurva PR
bukan 0.5** (tidak seperti ROC/AUC) - melainkan **proporsi kelas positif** dalam data
(proporsi hujan lebat). Model berguna jika kurvanya berada **di atas** garis baseline
putus-putus itu.


In [ ]:
from sklearn.metrics import precision_recall_curve
import os

# Kurva PR (Gambar 9.1): baseline acak = proporsi kelas positif, NIET 0.5!
prec, rec, _ = precision_recall_curve(leb, prob)
base = float(leb.mean())

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rec, prec, lw=2, label="Model (GRU)")
ax.axhline(base, color="gray", ls="--", lw=1.3,
           label=f"Baseline acak (proporsi positif {base:.0%})")
ax.set_xlabel("Recall (= POD)")
ax.set_ylabel("Precision (= 1 - FAR)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8.5)
plt.tight_layout()
plt.show()

# (Opsional, bila jalur lokal tersedia) salin ulang figur sebagai Gambar 9.1.
target = "../manuscripts/ch-09-studi-kasus-curah-hujan-terbuka/figures/fig-9-1-precision-recall.png"
if os.path.exists("../manuscripts"):
    fig.savefig(target, dpi=150)
    print("Figur ditaratin ke:", target)


## 7. Permutation Importance (Kode 9.2)

In [8]:
def perm_imp(model, X, y, n=5):
    # X: (n_samples, timesteps, n_features) — acak fitur j di semua timestep (Kode 9.2)
    base = mae(y, model.predict(X, verbose=0).ravel())
    imp = {}
    for j in range(X.shape[2]):
        scores = []
        for _ in range(n):
            Xp = X.copy()
            perm = np.random.default_rng((j, _)).permutation(Xp.shape[0])
            Xp[:, :, j] = Xp[perm, :, j]
            scores.append(mae(y, model.predict(Xp, verbose=0).ravel()))
        imp[feat[j]] = float(np.mean(scores) - base)
    return imp

imp = perm_imp(m, Xte, np.log1p(Yte))
pd.Series(imp).sort_values(ascending=False).round(4)

mus_cos     0.1643
hujan_t1    0.1265
mus_sin     0.1067
hujan_t7    0.0475
suhu        0.0427
rmm1       -0.0093
hujan_t2   -0.0168
nino34     -0.0232
hujan_t3   -0.0304
dtype: float64

## 7b. Tabel verifikasi per kategori intensitas

Satu angka CSI untuk kategori "lebat" tidak cukup: operasional butuh tahu bagaimanamodel berperilaku di **setiap kategori**. Berikut POD/FAR/CSI dihitung per kategori(one-vs-rest) menggunakan **prediksi regresi yang diubah ke kategori** — fairterhadap model regresi (bukan proba biner yang dilatih hanya untuk kategori lebat).

In [ ]:
def kategori_pred(mm):
    """Ubah jumlah hujan (mm) ke kategori, selaras dengan `kategori` di §2 (20/50 mm)."""
    return np.select([mm < 20, mm < 50], [0, 1], default=2)

pred_k = kategori_pred(pred)                  # dari model regresi (sel §5)
tabel = []
for k in [0, 1, 2]:
    yt = (Kte == k).astype(int)
    yp = (pred_k == k).astype(int)
    tp = ((yp == 1) & (yt == 1)).sum(); fp = ((yp == 1) & (yt == 0)).sum()
    fn = ((yp == 0) & (yt == 1)).sum()
    pod = tp / (tp + fn) if tp + fn else 0
    far = fp / (tp + fp) if tp + fp else 0
    csi = tp / (tp + fp + fn) if tp + fp + fn else 0
    tabel.append((k, pod, far, csi))

pd.DataFrame(tabel, columns=["kategori", "POD", "FAR", "CSI"]).round(3)


## 8. Latihan Mini

1. Tambahkan fitur regional nyata `era5_tp` (dari `data/era5/*/era5land_*_daily.csv` yang ter-commit, selaras dengan `kategori`) — lihat efek pada importance & MAE.
2. Bangun klasifikasi multi-kelas kategori (0/1/2) dengan `sparse_categorical_crossentropy`; buat crosstab.
3. Hitung CSI/POD/FAR per kategori dan bandingkan dengan Bab 9 Tabel 9.5.
4. Uji window `w ∈ {7, 14, 30}` di walk-forward 3 blok.
5. Ganti dengan station/koordinat lain (CHIRPS + ERA5/ERA5-Land + indeks MJO/ENSO via `scripts/download_*.py`) dan jalankan ulang.